# QUELL — Faz 1 / Step 01: Veri indirme + dogrulama

Bu notebook uc dataseti YPB'ye indirir ve her birini dogrular (file listesi, boyut,
rows sayisi, class distribution, hash). **JupyterHub'da hucre hucre calistir.**

Kaynaklar (hepsi CSV feature surumu — ham pcap DEGIL):
- **N-BaIoT** — UCI id 442 (yedek: Hugging Face `codymlewis/nbaiot`)
- **Edge-IIoTset** — Kaggle `mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot`
- **CICIoT2023** — resmi cicresearch.ca CSV dizini (recursive wget)

> Sorun cikan olursa o hucrenin ciktisini bana yapistir; yedek yolu birlikte devreye alalim.

In [ ]:
# 1) Paths and folders
import os, subprocess, sys, json, hashlib, glob
from pathlib import Path

ROOT = Path.home() / "quell-edge-llm-ids"   # change according to where your repo folder is
RAW  = ROOT / "data" / "raw"
for d in ["nbaiot", "edge_iiotset", "ciciot2023"]:
    (RAW / d).mkdir(parents=True, exist_ok=True)
print("Raw data folder:", RAW)
print(subprocess.run(f"df -h {RAW}", shell=True, capture_output=True, text=True).stdout)

## Kaggle kurulumu (Edge-IIoTset icin)
1. kaggle.com > hesabin > **Settings > API > Create New Token** → `kaggle.json` iner.
2. O fileyi JupyterHub'a yukle (home'a), sonra asagidaki hucre onu `~/.kaggle/`'a tasir.

Kaggle hesabin yoksa: ucretsiz, 2 dakika. Ya da bu dataseti elle indirip `data/raw/edge_iiotset/` icine koyabilirsin.

In [ ]:
# 2) Kaggle CLI + token
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=False)
kj_src = Path.home() / "kaggle.json"
kdir = Path.home() / ".kaggle"; kdir.mkdir(exist_ok=True)
if kj_src.exists():
    import shutil; shutil.copy(kj_src, kdir / "kaggle.json")
    os.chmod(kdir / "kaggle.json", 0o600)
    print("kaggle.json yerlestirildi.")
else:
    print("WARNING: kaggle.json not found in home. Upload it first, then run this cell again.")
print(subprocess.run("kaggle --version", shell=True, capture_output=True, text=True).stdout.strip() or "kaggle CLI not ready")

In [ ]:
# 3) Edge-IIoTset (Kaggle)
dst = RAW / "edge_iiotset"
r = subprocess.run(
    f'kaggle datasets download -d mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot -p "{dst}" --unzip',
    shell=True, capture_output=True, text=True)
print(r.stdout[-2000:]); print("STDERR:", r.stderr[-1000:])
print("--- downloaded files ---")
print(subprocess.run(f'find "{dst}" -maxdepth 3 -type f | head -50', shell=True, capture_output=True, text=True).stdout)

In [ ]:
# 4) CICIoT2023 — resmi CSV dizini (recursive). ~13 GB, zaman alabilir.
# Only .csv is downloaded; index.html files are discarded. The directory structure is discovered automatically (no hardcoded filename).
dst = RAW / "ciciot2023"
cmd = (f'wget -r -np -nH --cut-dirs=3 -R "index.html*" -A "csv,CSV" '
       f'-P "{dst}" "http://cicresearch.ca/IOTDataset/CIC_IOT_Dataset2023/Dataset/CSV/" 2>&1 | tail -20')
print("Starting (may take a while)...")
print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout)
print("--- inen csv sayisi ---")
print(subprocess.run(f'find "{dst}" -name "*.csv" | wc -l', shell=True, capture_output=True, text=True).stdout)
# NOT: if 0 files download, update the URL structure; check the output.

In [ ]:
# 5) N-BaIoT — UCI first, otherwise Hugging Face fallback
dst = RAW / "nbaiot"
uci = "https://archive.ics.uci.edu/static/public/442/detection+of+iot+botnet+attacks+n+baiot.zip"
r = subprocess.run(f'wget -q --show-progress -O "{dst}/nbaiot.zip" "{uci}"',
                   shell=True, capture_output=True, text=True)
ok = (dst / "nbaiot.zip").exists() and (dst / "nbaiot.zip").stat().st_size > 10_000
if ok:
    subprocess.run(f'cd "{dst}" && unzip -o -q nbaiot.zip', shell=True)
    print("Downloaded and extracted from UCI.")
else:
    print("UCI yolu tutmadi, Hugging Face yedegine geciliyor...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=False)
    r2 = subprocess.run(
        f'cd "{dst}" && git clone https://huggingface.co/datasets/codymlewis/nbaiot hf_nbaiot',
        shell=True, capture_output=True, text=True)
    print(r2.stdout[-500:], r2.stderr[-500:])
print(subprocess.run(f'find "{dst}" -type f | head -40', shell=True, capture_output=True, text=True).stdout)

## Dogrulama
Her dataset icin: kac file, toplam boyut, ornek filenin sekli (rows x column) ve class distribution.
Sonuclar `results/data_manifest.json`'a yazilir (reproducibility).

In [ ]:
# 6) Dogrulama + manifest
import pandas as pd
def sha256(p, cap=200*1024*1024):
    h=hashlib.sha256(); n=0
    with open(p,'rb') as f:
        for ch in iter(lambda:f.read(1<<20),b''):
            h.update(ch); n+=len(ch)
            if n>=cap: break
    return h.hexdigest()[:16]

manifest={}
for name in ["nbaiot","edge_iiotset","ciciot2023"]:
    d=RAW/name
    csvs=sorted(glob.glob(str(d/"**/*.csv"),recursive=True))
    total=sum(os.path.getsize(c) for c in csvs)
    info={"num_csv":len(csvs),"total_MB":round(total/1024**2,1),
          "example_file":os.path.relpath(csvs[0],RAW) if csvs else None}
    if csvs:
        try:
            df=pd.read_csv(csvs[0],nrows=50000,low_memory=False)
            info["example_shape"]=list(df.shape)
            info["columns_head"]=list(df.columns[:8])
            lab=[c for c in df.columns if c.lower() in ("label","attack","type","class","category")]
            if lab: info["label_col"]=lab[0]; info["label_values_sample"]=df[lab[0]].value_counts().head(10).to_dict()
            info["sha256_16"]=sha256(csvs[0])
        except Exception as e:
            info["read_error"]=str(e)
    manifest[name]=info
    print(f"\n=== {name} ==="); print(json.dumps(info,indent=2,default=str,ensure_ascii=False))

(ROOT/"results").mkdir(exist_ok=True)
json.dump(manifest,open(ROOT/"results"/"data_manifest.json","w"),indent=2,default=str,ensure_ascii=False)
print("\nManifest yazildi -> results/data_manifest.json")